# Training a retrieval network here, against the published one

`cbir train` fine-tunes a backbone on retrieval-SfM-120k and `cbir whiten` fits the
supervised projection that finishes it. This notebook asks the only question that
matters about that: **does a checkpoint we trained land where the published one does?**

Everything plotted here was measured by this repository through one evaluation path.
The two published GeM rows are the exception and are drawn as reference rules, not bars,
so a measured number is never confused with a quoted one. Figures for other methods in
the literature are deliberately absent: this project scores what it can re-run.

All mAP on 0-1. Benchmark is roxford5k, multi-scale, held-out training corpus.

In [ ]:
import json
import re
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import style as sty

from cbir.eval.frame import tidy_rows
from cbir.eval.results import latest

sty.use()
PROTOCOLS = sty.PROTOCOLS
DOCS = Path("..") / "docs"
DOCS.mkdir(exist_ok=True)


def short(run):
    # `vgg16-gem-margin0.7-lr1e-06-seed0` -> `lr 1e-06`, which is what varies here.
    # Anchored on `-seed`, not on the next hyphen: the exponent contains one.
    found = re.search(r"-lr(.+?)-seed", run)
    return f"lr {found.group(1)}" if found else run

## The data

One row per (run x protocol), with the parameters that identify a checkpoint promoted to columns.

In [ ]:
df = pd.DataFrame(tidy_rows(records=list(latest().values())))
params = df["params"]
df["backbone"] = params.apply(lambda d: d.get("backbone") or "")
df["weights"] = params.apply(lambda d: d.get("weights") or "")
df["whiten_source"] = params.apply(lambda d: d.get("whiten_source") or "")
df["checkpoint"] = params.apply(lambda d: d.get("checkpoint") or "")
df["scales"] = params.apply(lambda d: len(d["scales"]) if isinstance(d.get("scales"), list) else 1)
# A checkpoint path means we trained it; its absence means the published file.
df["provenance"] = df["checkpoint"].apply(lambda s: "trained here" if s else "published")
df["run"] = df["checkpoint"].apply(lambda s: s.split("/")[-2] if "/" in s else "published")

FT = df[(df["technique"] == "gem") & (df["weights"] == "sfm120k") & (df["scales"] == 3)]
FT[["backbone", "run", "whiten_source", "protocol", "map"]].sort_values(
    ["backbone", "run", "whiten_source", "protocol"]
).reset_index(drop=True)

## Ours against the published checkpoint

Both whitened by their own fitted projection, so the comparison isolates the network.
The published bar is the reference; ours is the question.

In [ ]:
def best(backbone, provenance, whiten_source="learned"):
    rows = FT[(FT["backbone"] == backbone) & (FT["provenance"] == provenance)
              & (FT["whiten_source"] == whiten_source)]
    return {p: rows[rows["protocol"] == p]["map"].max() for p in PROTOCOLS}

BACKBONES = ["vgg16", "resnet101"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), sharey=True)
for ax, backbone in zip(axes, BACKBONES):
    published, ours = best(backbone, "published"), best(backbone, "trained here")
    x = range(len(PROTOCOLS))
    for offset, (name, values) in zip((-0.19, 0.19), (("published", published), ("trained here", ours))):
        bars = ax.bar([i + offset for i in x], [values[p] for p in PROTOCOLS], 0.36,
                      color=sty.PROVENANCE[name], label=name, zorder=3,
                      edgecolor=sty.SURFACE, linewidth=1.2)
        for bar, p in zip(bars, PROTOCOLS):
            sty.label_point(ax, bar.get_x() + bar.get_width() / 2, bar.get_height(),
                            f"{values[p]:.3f}", dx=0, dy=5, ha="center", fontsize=7.5)
    ax.set_xticks(list(x))
    ax.set_xticklabels([p.capitalize() for p in PROTOCOLS])
    sty.style(ax, title=backbone, ylabel="mAP" if backbone == BACKBONES[0] else None)
    ax.set_ylim(0, 1.0)
axes[0].legend(frameon=False, fontsize=9, loc="upper right")
fig.suptitle("Fine-tuned GeM on roxford5k, multi-scale, supervised whitening",
             x=0.005, ha="left", fontsize=11.5, color=sty.INK)
fig.tight_layout()
fig.savefig(DOCS / "trained_vs_published.png", bbox_inches="tight", facecolor=sty.SURFACE)

## What the supervised whitening is worth

`cbir train` produces the network; `cbir whiten` fits the projection applied to its
output. The same checkpoint read through held-out PCA instead is the counterfactual --
the difference is what the labels bought, with the weights held fixed.

In [ ]:
pairs = []
for backbone in BACKBONES:
    for provenance in ("published", "trained here"):
        for run in sorted(FT[(FT["backbone"] == backbone) & (FT["provenance"] == provenance)]["run"].unique()):
            rows = FT[(FT["backbone"] == backbone) & (FT["run"] == run)]
            have = set(rows["whiten_source"])
            if {"held_out", "learned"} <= have:
                pairs.append((backbone, run, provenance, rows))

fig, axes = plt.subplots(1, len(pairs), figsize=(3.4 * len(pairs), 3.9), sharey=True, squeeze=False)
for ax, (backbone, run, provenance, rows) in zip(axes[0], pairs):
    for source, hatch in (("held_out", "//"), ("learned", None)):
        values = [rows[(rows["protocol"] == p) & (rows["whiten_source"] == source)]["map"].max() for p in PROTOCOLS]
        offset = -0.19 if source == "held_out" else 0.19
        ax.bar([i + offset for i in range(len(PROTOCOLS))], values, 0.36,
               color=sty.PROVENANCE[provenance], hatch=hatch, edgecolor=sty.SURFACE, linewidth=1.5,
               alpha=0.45 if source == "held_out" else 1.0,
               label="held-out PCA" if source == "held_out" else "supervised", zorder=3)
    gains = [rows[(rows["protocol"] == p) & (rows["whiten_source"] == "learned")]["map"].max()
             - rows[(rows["protocol"] == p) & (rows["whiten_source"] == "held_out")]["map"].max()
             for p in PROTOCOLS]
    for i, gain in enumerate(gains):
        sty.label_point(ax, i, 0.03, f"+{gain:.3f}", dx=0, ha="center", fontsize=8, color=sty.INK)
    ax.set_xticks(range(len(PROTOCOLS)))
    ax.set_xticklabels([p.capitalize() for p in PROTOCOLS])
    subtitle = provenance if provenance == "published" else f"{provenance}, {short(run)}"
    sty.style(ax, title=f"{backbone}\n{subtitle}", ylabel="mAP" if ax is axes[0][0] else None)
    ax.set_ylim(0, 1.0)
axes[0][0].legend(frameon=False, fontsize=8.5, loc="upper right")
fig.suptitle("Supervised whitening against held-out PCA, weights held fixed",
             x=0.005, ha="left", fontsize=11.5, color=sty.INK)
fig.tight_layout()
fig.savefig(DOCS / "trained_whitening.png", bbox_inches="tight", facecolor=sty.SURFACE)

## The whole ladder on Medium

Every step this project has taken on one axis, best row per stage. The published
fine-tuned checkpoints are drawn as rules rather than bars -- they are the target, not
another entry.

In [ ]:
def top(mask, protocol):
    rows = df[mask & (df["protocol"] == protocol)]
    return rows["map"].max() if len(rows) else float("nan")

classic = df["technique"].isin(["bow", "vlad", "fisher"])
shelf = (df["technique"].isin(["gem", "rmac"])) & (df["weights"] != "sfm120k")
LADDER = [
    ("classic\n(VLAD)", classic, "classic"),
    ("off-the-shelf\nCNN", shelf, "off-the-shelf CNN"),
    ("trained here\nvgg16", (df["run"] == "vgg16-gem-margin0.7-lr1e-06-seed0") & (df["whiten_source"] == "learned"), "fine-tuned CNN"),
    ("trained here\nresnet101", (df["run"] == "resnet101-gem-margin0.85-lr5e-07-seed0") & (df["whiten_source"] == "learned"), "fine-tuned CNN"),
]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), sharey=True)
for ax, protocol in zip(axes, ["medium", "hard"]):
    values = [top(mask, protocol) for _, mask, _ in LADDER]
    bars = ax.bar(range(len(LADDER)), values, 0.6,
                  color=[sty.TIER[tier] for _, _, tier in LADDER], zorder=3)
    for bar, value in zip(bars, values):
        sty.label_point(ax, bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}",
                        dx=0, dy=5, ha="center", fontsize=8.5, color=sty.INK)
    for backbone, dash in (("vgg16", (4, 3)), ("resnet101", (1, 2))):
        target = best(backbone, "published")[protocol]
        ax.axhline(target, color=sty.MUTED, linewidth=1.1, dashes=dash, zorder=2)
        # Anchored to the left edge inside the axes: at the right edge the text was
        # clipped away entirely, leaving two dash styles with nothing to identify them.
        sty.label_point(ax, -0.45, target, f"published {backbone}", dx=0, dy=6,
                        ha="left", fontsize=7.5)
    ax.set_xticks(range(len(LADDER)))
    ax.set_xticklabels([name for name, _, _ in LADDER], fontsize=8.5)
    sty.style(ax, title=protocol.capitalize(), ylabel="mAP" if protocol == "medium" else None)
    ax.set_ylim(0, 0.78)
fig.suptitle("roxford5k: best measured row per stage, against the published targets",
             x=0.005, ha="left", fontsize=11.5, color=sty.INK)
fig.tight_layout()
fig.savefig(DOCS / "trained_ladder.png", bbox_inches="tight", facecolor=sty.SURFACE)

## How the two runs got there

Validation mAP per epoch, read from the trackio store the training loop writes. This is
the held-out SfM landmark split, **not** roxford5k -- a different corpus and a different
label definition, so it ranks checkpoints during a run and nothing more.

In [ ]:
STORE = Path.home() / ".cache" / "huggingface" / "trackio" / "cbir-train.db"

def curves():
    if not STORE.exists():
        return {}
    with sqlite3.connect(STORE) as connection:
        frame = pd.read_sql("select run_name, step, metrics from metrics", connection)
    # The column is a JSON blob, stored as bytes.
    def value(blob):
        payload = json.loads(blob.decode() if isinstance(blob, bytes) else blob)
        return payload.get("val/mean_average_precision")

    frame["map"] = frame["metrics"].apply(value)
    frame = frame.dropna(subset=["map"])
    return {name: group.sort_values("step") for name, group in frame.groupby("run_name")}

series = curves()
if not series:
    print("no trackio store at", STORE, "- skipping")
else:
    fig, ax = plt.subplots(figsize=(7.5, 4.0))
    # One slot per run, not per backbone: the two vgg16 runs previously shared both
    # colour and dash and were distinguishable only by reading the labels.
    slots = list(sty.TECHNIQUE.values())
    names = sorted(series)
    for slot, name in zip(slots, names):
        group = series[name]
        backbone = "resnet101" if "resnet101" in name else "vgg16"
        ax.plot(group["step"], group["map"], color=slot, linewidth=2, zorder=3,
                label=f"{backbone}, {short(name)}")
        last = group.iloc[-1]
        sty.label_point(ax, last["step"], last["map"], f"{backbone} {short(name)}", dx=6, fontsize=7.5)
    sty.style(ax, title="Validation mAP per epoch (SfM landmark split, not roxford5k)",
              xlabel="epoch", ylabel="val mAP")
    # Headroom on the right, or the direct labels run off the canvas.
    ax.set_xlim(0, max(g["step"].max() for g in series.values()) * 1.32)
    ax.legend(frameon=False, fontsize=8.5, loc="lower right")
    fig.tight_layout()
    fig.savefig(DOCS / "trained_curves.png", bbox_inches="tight", facecolor=sty.SURFACE)